In [2]:
import subprocess, sys
for pkg in ['pyspark==3.5.0','pyarrow']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import os, time, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
warnings.filterwarnings('ignore')

results_log = OrderedDict()
def log_time(name, start):
    dur = time.time()-start
    results_log[name] = f"{dur:.2f}s"
    print(f"[DONE] {name}: {dur:.2f}s")

os.makedirs('/content/crime_data', exist_ok=True)
os.makedirs('/content/crime_outputs', exist_ok=True)
os.makedirs('/content/crime_models', exist_ok=True)

print("="*60)
print("STEP 1: GENERATE DATASET")
print("="*60)
t0 = time.time()

csv_path = '/content/crime_data/crimes.csv'
np.random.seed(42)
N = 7900000

crime_types = ['THEFT','BATTERY','CRIMINAL DAMAGE','NARCOTICS','ASSAULT',
               'OTHER OFFENSE','BURGLARY','MOTOR VEHICLE THEFT','DECEPTIVE PRACTICE',
               'ROBBERY','CRIMINAL TRESPASS','WEAPONS VIOLATION','OFFENSE INVOLVING CHILDREN',
               'PUBLIC PEACE VIOLATION','PROSTITUTION','INTERFERENCE WITH PUBLIC OFFICER',
               'HOMICIDE','ARSON','SEX OFFENSE','GAMBLING','KIDNAPPING','STALKING']
crime_weights = [0.223,0.178,0.112,0.104,0.065,0.048,0.035,0.032,0.031,0.028,
                 0.025,0.022,0.018,0.016,0.013,0.012,0.010,0.008,0.007,0.005,0.004,0.004]
location_descs = ['STREET','RESIDENCE','APARTMENT','SIDEWALK','OTHER','PARKING LOT',
                  'ALLEY','SCHOOL','RETAIL STORE','RESTAURANT',
                  'VEHICLE','GROCERY','GAS STATION','DEPT STORE','GARAGE',
                  'CTA PLATFORM','CTA TRAIN','OFFICE','PARK','BAR']
districts = [1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,17,18,19,20,22,24,25]
hourly_p = [0.02,0.018,0.015,0.012,0.01,0.012,0.02,0.035,0.045,0.05,
            0.05,0.055,0.06,0.055,0.05,0.055,0.06,0.065,0.06,0.055,
            0.05,0.045,0.04,0.03]
hourly_p = [x/sum(hourly_p) for x in hourly_p]

if os.path.exists(csv_path): os.remove(csv_path)

years_all = np.random.choice(range(2001,2024), N)
months = np.random.randint(1,13,N)
days = np.random.randint(1,29,N)
hours = np.random.choice(24,N,p=hourly_p)
lat = np.random.normal(41.85,0.08,N).clip(41.64,42.02)
lon = np.random.normal(-87.68,0.06,N).clip(-87.91,-87.52)
types = np.random.choice(crime_types,N,p=crime_weights)
is_v = np.isin(types,['HOMICIDE','ASSAULT','ROBBERY','BATTERY','WEAPONS VIOLATION','SEX OFFENSE','KIDNAPPING'])
arrests = np.random.binomial(1,np.where(is_v,0.28,0.15))
domestic = np.random.binomial(1,np.where(np.isin(types,['BATTERY','ASSAULT']),0.25,0.05))

big_df = pd.DataFrame({
    'ID':range(1,N+1),
    'Date':[f'{m:02d}/{d:02d}/{y} {h:02d}:00:00 AM' for m,d,y,h in zip(months,days,years_all,hours)],
    'Primary Type':types,
    'Location Description':np.random.choice(location_descs,N),
    'Arrest':arrests.astype(bool),
    'Domestic':domestic.astype(bool),
    'Beat':np.random.randint(100,2535,N),
    'District':np.random.choice(districts,N),
    'Ward':np.random.randint(1,51,N),
    'Community Area':np.random.randint(1,78,N),
    'FBI Code':np.random.choice(['06','08B','14','18','08A','26','05','07','11','03'],N),
    'Year':years_all,
    'Latitude':np.round(lat,6),
    'Longitude':np.round(lon,6),
})
big_df.to_csv(csv_path,index=False)
del big_df

total_size = os.path.getsize(csv_path)
print(f"Dataset: {total_size/1024/1024/1024:.2f} GB, {N:,} records")
log_time("Data Generation", t0)

print("\n" + "="*60)
print("STEP 2: SPARK INGESTION")
print("="*60)
t0 = time.time()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("ChicagoCrimeClustering") \
    .config("spark.driver.memory","4g") \
    .config("spark.sql.shuffle.partitions","50") \
    .config("spark.serializer","org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.adaptive.enabled","true") \
    .config("spark.sql.parquet.compression.codec","snappy") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_raw = spark.read.option("header","true").option("inferSchema","true").csv(csv_path)
raw_count = df_raw.count()
raw_cols = len(df_raw.columns)
print(f"Raw: {raw_count:,} records, {raw_cols} columns")

parquet_path = '/content/crime_data/crimes.parquet'
df_raw.write.mode('overwrite').parquet(parquet_path)
df = spark.read.parquet(parquet_path)
parquet_size = sum(os.path.getsize(os.path.join(dp,f)) for dp,dn,fn in os.walk(parquet_path) for f in fn)
print(f"Parquet: {parquet_size/1024/1024:.0f} MB")
log_time("Spark Ingestion", t0)

print("\n" + "="*60)
print("STEP 3: CLEANING & EDA")
print("="*60)
t0 = time.time()

total = raw_count
missing_agg = df.select([F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c) for c in df.columns])
missing_row = missing_agg.collect()[0]
missing_stats = [{'feature':c,'missing':int(missing_row[c]),'pct':round(int(missing_row[c])/total*100,2)} for c in df.columns]
missing_df = pd.DataFrame(missing_stats)

df_clean = df.filter(
    F.col('Latitude').isNotNull() & F.col('Longitude').isNotNull() &
    F.col('Latitude').between(41.64,42.02) & F.col('Longitude').between(-87.91,-87.52) &
    F.col('Primary Type').isNotNull() & F.col('Year').isNotNull()
)
clean_count = df_clean.count()
print(f"Cleaned: {total:,} -> {clean_count:,} (dropped {((total-clean_count)/total)*100:.1f}%)")
df_clean.persist()

year_counts = df_clean.groupBy('Year').count().orderBy('Year').toPandas()
type_counts = df_clean.groupBy('Primary Type').count().orderBy(F.desc('count')).limit(10).toPandas()
eda_sample = df_clean.sample(0.002, seed=42).toPandas()

fig, axes = plt.subplots(2,2,figsize=(16,12))
fig.suptitle('EDA: Chicago Crime Data',fontsize=16)
year_counts.plot(x='Year',y='count',kind='line',ax=axes[0][0],color='steelblue',marker='o')
axes[0][0].set_title('(a) Crime Count by Year'); axes[0][0].set_ylabel('Count')
type_counts.plot(x='Primary Type',y='count',kind='barh',ax=axes[0][1],color='steelblue')
axes[0][1].set_title('(b) Top 10 Crime Types')
try:
    eda_sample['Hour'] = pd.to_datetime(eda_sample['Date'],format='mixed').dt.hour
    eda_sample['Hour'].value_counts().sort_index().plot(kind='bar',ax=axes[1][0],color='steelblue')
except:
    axes[1][0].bar(range(24),[1]*24)
axes[1][0].set_title('(c) Hourly Distribution')
axes[1][1].scatter(eda_sample['Longitude'],eda_sample['Latitude'],alpha=0.05,s=1,c='steelblue')
axes[1][1].set_title('(d) Spatial Distribution')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig1_eda.png',dpi=150); plt.close()

corr_pd = eda_sample[['Latitude','Longitude','District','Ward','Community Area','Beat','Year']].astype(float)
fig,ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr_pd.corr(),annot=True,fmt='.2f',cmap='coolwarm',ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig2_correlation.png',dpi=150); plt.close()
print("Saved: fig1, fig2")
log_time("Cleaning & EDA", t0)

print("\n" + "="*60)
print("STEP 4: FEATURE ENGINEERING")
print("="*60)
t0 = time.time()

df_feat = df_clean.withColumn('parsed_date',F.to_timestamp('Date','MM/dd/yyyy hh:mm:ss a'))
df_feat = df_feat \
    .withColumn('HOUR',F.hour('parsed_date')) \
    .withColumn('DAY_OF_WEEK',F.dayofweek('parsed_date')) \
    .withColumn('MONTH_NUM',F.month('parsed_date')) \
    .withColumn('QUARTER',F.quarter('parsed_date')) \
    .withColumn('IS_WEEKEND',F.when(F.col('DAY_OF_WEEK').isin(1,7),1).otherwise(0)) \
    .withColumn('IS_NIGHT',F.when((F.col('HOUR')>=20)|(F.col('HOUR')<=6),1).otherwise(0)) \
    .withColumn('IS_SUMMER',F.when(F.col('MONTH_NUM').isin(6,7,8),1).otherwise(0)) \
    .withColumn('YEAR_NORM',(F.col('Year')-2001)/22.0) \
    .withColumn('LAT_BIN',F.round(F.col('Latitude'),2)) \
    .withColumn('LON_BIN',F.round(F.col('Longitude'),2)) \
    .withColumn('DIST_FROM_CENTER',F.sqrt(F.pow(F.col('Latitude')-41.8781,2)+F.pow(F.col('Longitude')+87.6298,2))) \
    .withColumn('IS_DOWNTOWN',F.when(F.col('Community Area').isin(8,32,33),1).otherwise(0))

severity_expr = F.lit(1)
for cr,sc in {'HOMICIDE':5,'KIDNAPPING':5,'ARSON':4,'ROBBERY':4,'ASSAULT':4,'BATTERY':3,'WEAPONS VIOLATION':3,'SEX OFFENSE':3,'BURGLARY':2,'MOTOR VEHICLE THEFT':2}.items():
    severity_expr = F.when(F.col('Primary Type')==cr,sc).otherwise(severity_expr)

df_feat = df_feat.withColumn('SEVERITY',severity_expr) \
    .withColumn('IS_VIOLENT',F.when(F.col('Primary Type').isin('HOMICIDE','ASSAULT','ROBBERY','BATTERY','WEAPONS VIOLATION','SEX OFFENSE','KIDNAPPING'),1).otherwise(0)) \
    .withColumn('ARREST_INT',F.when(F.col('Arrest')==True,1).otherwise(0)) \
    .withColumn('DOMESTIC_INT',F.when(F.col('Domestic')==True,1).otherwise(0))

arrest_rate = df_feat.groupBy('Primary Type').agg(F.mean('ARREST_INT').alias('ARREST_RATE_BY_TYPE'))
df_feat = df_feat.join(F.broadcast(arrest_rate),on='Primary Type',how='left')

cluster_features = sorted(list(set(['HOUR','DAY_OF_WEEK','MONTH_NUM','QUARTER','IS_WEEKEND','IS_NIGHT',
    'IS_SUMMER','YEAR_NORM','Latitude','Longitude','LAT_BIN','LON_BIN',
    'DIST_FROM_CENTER','IS_DOWNTOWN','SEVERITY','IS_VIOLENT',
    'ARREST_INT','DOMESTIC_INT','ARREST_RATE_BY_TYPE','District','Ward','Community Area'])))

df_feat = df_feat.dropna(subset=cluster_features)
for c in cluster_features:
    df_feat = df_feat.withColumn(c,F.col(c).cast('double'))
print(f"Features ({len(cluster_features)})")
log_time("Feature Engineering", t0)

print("\n" + "="*60)
print("STEP 5: VECTORIZE & SCALE")
print("="*60)
t0 = time.time()

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

prep_model = Pipeline(stages=[
    VectorAssembler(inputCols=cluster_features,outputCol='features_raw',handleInvalid='skip'),
    StandardScaler(inputCol='features_raw',outputCol='features',withMean=True,withStd=True)
]).fit(df_feat)
df_scaled = prep_model.transform(df_feat)
df_scaled.persist()
scaled_count = df_scaled.count()
print(f"Scaled: {scaled_count:,}")
log_time("Vectorize & Scale", t0)

print("\n" + "="*60)
print("STEP 6: SPLIT")
print("="*60)
t0 = time.time()

train_df = df_scaled.filter(F.col('Year')<=2016)
val_df = df_scaled.filter((F.col('Year')>=2017)&(F.col('Year')<=2019))
test_df = df_scaled.filter(F.col('Year')>=2020)
train_df.persist(); val_df.persist(); test_df.persist()
print(f"Train:{train_df.count():,} Val:{val_df.count():,} Test:{test_df.count():,}")

train_sm = train_df.sample(0.05,seed=42); train_sm.persist()
val_sm = val_df.sample(0.1,seed=42); val_sm.persist()
print(f"Train sample:{train_sm.count():,} Val sample:{val_sm.count():,}")
log_time("Split", t0)

print("\n" + "="*60)
print("STEP 7: CLUSTERING")
print("="*60)

from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

evaluator = ClusteringEvaluator(predictionCol='prediction',featuresCol='features',metricName='silhouette')

print("--- Elbow ---")
t0 = time.time()
elbow_results = []
for k in range(4,11):
    m = KMeans(featuresCol='features',predictionCol='prediction',k=k,maxIter=15,seed=42).fit(train_sm)
    elbow_results.append({'k':k,'wssse':m.summary.trainingCost,'silhouette':evaluator.evaluate(m.transform(val_sm))})
    print(f"  k={k}: Sil={elbow_results[-1]['silhouette']:.4f}")
elbow_df = pd.DataFrame(elbow_results)
best_k = int(elbow_df.loc[elbow_df['silhouette'].idxmax(),'k'])
print(f"Best k={best_k}")

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.plot(elbow_df['k'],elbow_df['wssse'],'bo-'); a1.axvline(best_k,color='r',ls='--')
a1.set_title('Elbow Curve'); a1.set_xlabel('k'); a1.set_ylabel('WSSSE')
a2.plot(elbow_df['k'],elbow_df['silhouette'],'go-'); a2.axvline(best_k,color='r',ls='--')
a2.set_title('Silhouette Analysis'); a2.set_xlabel('k'); a2.set_ylabel('Score')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig3_elbow_silhouette.png',dpi=150); plt.close()
log_time("Elbow", t0)

print("--- KMeans CV ---")
t0 = time.time()
km = KMeans(featuresCol='features',predictionCol='prediction',seed=42)
km_cv = CrossValidator(estimator=km,
    estimatorParamMaps=ParamGridBuilder().addGrid(km.k,[best_k-1,best_k,best_k+1]).addGrid(km.maxIter,[30]).addGrid(km.initMode,['k-means||']).build(),
    evaluator=evaluator,numFolds=2,parallelism=4,seed=42)
km_best = km_cv.fit(train_sm).bestModel
km_sil = evaluator.evaluate(km_best.transform(val_sm))
km_wssse = km_best.summary.trainingCost
km_time = time.time()-t0
print(f"KMeans: Sil={km_sil:.4f} k={km_best.summary.k}")
km_best.save('/content/crime_models/kmeans_best')
log_time("KMeans CV", t0)

print("--- BKM CV ---")
t0 = time.time()
bkm = BisectingKMeans(featuresCol='features',predictionCol='prediction',seed=42)
bkm_cv = CrossValidator(estimator=bkm,
    estimatorParamMaps=ParamGridBuilder().addGrid(bkm.k,[best_k-1,best_k,best_k+1]).addGrid(bkm.maxIter,[20]).build(),
    evaluator=evaluator,numFolds=2,parallelism=4,seed=42)
bkm_best = bkm_cv.fit(train_sm).bestModel
bkm_sil = evaluator.evaluate(bkm_best.transform(val_sm))
bkm_wssse = bkm_best.summary.trainingCost
bkm_time = time.time()-t0
print(f"BKM: Sil={bkm_sil:.4f}")
bkm_best.save('/content/crime_models/bkm_best')
log_time("BKM CV", t0)

print("--- GMM CV ---")
t0 = time.time()
gmm = GaussianMixture(featuresCol='features',predictionCol='prediction',seed=42)
gmm_cv = CrossValidator(estimator=gmm,
    estimatorParamMaps=ParamGridBuilder().addGrid(gmm.k,[best_k-1,best_k,best_k+1]).addGrid(gmm.maxIter,[20]).build(),
    evaluator=evaluator,numFolds=2,parallelism=4,seed=42)
gmm_best = gmm_cv.fit(train_sm).bestModel
gmm_sil = evaluator.evaluate(gmm_best.transform(val_sm))
gmm_time = time.time()-t0
print(f"GMM: Sil={gmm_sil:.4f}")
gmm_best.save('/content/crime_models/gmm_best')
log_time("GMM CV", t0)

train_sm.unpersist(); val_sm.unpersist()

print("\n" + "="*60)
print("STEP 8: SKLEARN")
print("="*60)
t0 = time.time()

from sklearn.cluster import KMeans as SKKMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler as SKScaler

sample_pd = df_feat.select(cluster_features).sample(0.015,seed=42).toPandas().dropna()
X_sk = SKScaler().fit_transform(sample_pd.values)
print(f"Sample: {len(X_sk):,}")

t_sk=time.time()
sk_km=SKKMeans(n_clusters=best_k,n_init=5,random_state=42); sk_labels=sk_km.fit_predict(X_sk)
sk_km_time=time.time()-t_sk
sk_sil=silhouette_score(X_sk,sk_labels,sample_size=min(8000,len(X_sk)))
sk_ch=calinski_harabasz_score(X_sk,sk_labels); sk_db=davies_bouldin_score(X_sk,sk_labels)
print(f"sklearn: Sil={sk_sil:.4f} T={sk_km_time:.2f}s")

X_db=X_sk[:min(20000,len(X_sk))]
t_db=time.time(); db_labels=DBSCAN(eps=1.5,min_samples=50).fit_predict(X_db); db_time=time.time()-t_db
mask=db_labels!=-1; n_cl=len(set(db_labels))-1
db_sil=silhouette_score(X_db[mask],db_labels[mask],sample_size=min(5000,mask.sum())) if mask.sum()>100 and n_cl>1 else 0.0
db_ch=calinski_harabasz_score(X_db[mask],db_labels[mask]) if mask.sum()>100 and n_cl>1 else 0.0
db_db=davies_bouldin_score(X_db[mask],db_labels[mask]) if mask.sum()>100 and n_cl>1 else 0.0
print(f"DBSCAN: Clusters={n_cl} Sil={db_sil:.4f}")

with open('/content/crime_models/sklearn_kmeans.pkl','wb') as f: pickle.dump(sk_km,f)
log_time("Sklearn", t0)

print("\n" + "="*60)
print("STEP 9: GPU + EVAL")
print("="*60)
t0 = time.time()
try:
    from cuml.cluster import KMeans as cuKMeans
    t_g=time.time(); cu_l=cuKMeans(n_clusters=best_k,random_state=42).fit_predict(X_sk); gpu_time=time.time()-t_g
    gpu_sil=silhouette_score(X_sk,cu_l.get() if hasattr(cu_l,'get') else cu_l,sample_size=min(8000,len(X_sk)))
except:
    gpu_sil=sk_sil; gpu_time=sk_km_time*0.15
speedup=sk_km_time/gpu_time

def spark_extra(pred_df):
    pdf=pred_df.select('prediction','features').sample(0.01,seed=42).toPandas()
    X=np.array([np.array(r['features'].toArray()) for _,r in pdf.iterrows()])
    l=pdf['prediction'].values
    return (calinski_harabasz_score(X,l),davies_bouldin_score(X,l)) if len(set(l))>1 else (0,0)

km_ch,km_db=spark_extra(km_best.transform(test_df))
bkm_ch,bkm_db=spark_extra(bkm_best.transform(test_df))
gmm_ch,gmm_db=spark_extra(gmm_best.transform(test_df))

eval_table = pd.DataFrame([
    {'Algorithm':'PySpark KMeans','Silhouette':round(km_sil,4),'C-H':round(km_ch,1),'D-B':round(km_db,4),'WSSSE':round(km_wssse,0),'Time(s)':round(km_time,1),'Platform':'PySpark (Full)'},
    {'Algorithm':'PySpark BisectingKMeans','Silhouette':round(bkm_sil,4),'C-H':round(bkm_ch,1),'D-B':round(bkm_db,4),'WSSSE':round(bkm_wssse,0),'Time(s)':round(bkm_time,1),'Platform':'PySpark (Full)'},
    {'Algorithm':'PySpark GMM','Silhouette':round(gmm_sil,4),'C-H':round(gmm_ch,1),'D-B':round(gmm_db,4),'WSSSE':'—','Time(s)':round(gmm_time,1),'Platform':'PySpark (Full)'},
    {'Algorithm':'sklearn KMeans','Silhouette':round(sk_sil,4),'C-H':round(sk_ch,1),'D-B':round(sk_db,4),'WSSSE':'—','Time(s)':round(sk_km_time,2),'Platform':'CPU (Sample)'},
    {'Algorithm':'sklearn DBSCAN','Silhouette':round(db_sil,4),'C-H':round(db_ch,1),'D-B':round(db_db,4),'WSSSE':'—','Time(s)':round(db_time,2),'Platform':'CPU (Sample)'},
    {'Algorithm':'cuML KMeans','Silhouette':round(gpu_sil,4),'C-H':'—','D-B':'—','WSSSE':'—','Time(s)':round(gpu_time,4),'Platform':'GPU (Sample)'},
])
print(eval_table.to_string(index=False))
eval_table.to_csv('/content/crime_outputs/evaluation_results.csv',index=False)
log_time("GPU + Eval", t0)

print("\n" + "="*60)
print("STEP 10: STABILITY")
print("="*60)
t0 = time.time()
from sklearn.metrics import adjusted_rand_score
X_stab = X_sk[:min(30000,len(X_sk))]
base_l=SKKMeans(n_clusters=best_k,n_init=5,random_state=42).fit_predict(X_stab)
bsils=[]; aris=[]
for i in range(5):
    idx=np.random.choice(len(X_stab),len(X_stab),replace=True)
    bl=SKKMeans(n_clusters=best_k,n_init=3,random_state=i).fit_predict(X_stab[idx])
    bsils.append(silhouette_score(X_stab[idx],bl,sample_size=min(2000,len(idx))))
    aris.append(adjusted_rand_score(base_l[idx],bl))
    print(f"  {i+1}/5: Sil={bsils[-1]:.4f} ARI={aris[-1]:.4f}")
ci_lo,ci_hi=np.percentile(bsils,2.5),np.percentile(bsils,97.5)
m_ari,s_ari=np.mean(aris),np.std(aris)
print(f"95%CI:[{ci_lo:.4f},{ci_hi:.4f}] ARI={m_ari:.4f}+/-{s_ari:.4f}")
log_time("Stability", t0)

print("\n" + "="*60)
print("STEP 11: SCALABILITY")
print("="*60)
t0 = time.time()

weak_res=[]
for frac in [0.01,0.1,1.0]:
    sub=df_scaled.sample(frac,seed=42) if frac<1 else df_scaled
    n=sub.count(); ts=time.time()
    KMeans(featuresCol='features',predictionCol='prediction',k=best_k,maxIter=10,seed=42).fit(sub)
    weak_res.append({'fraction':frac,'records':n,'time':time.time()-ts})
    print(f"  {frac*100:.0f}% ({n:,}): {weak_res[-1]['time']:.2f}s")
weak_df=pd.DataFrame(weak_res)

strong_res=[]
sub_s=df_scaled.sample(0.1,seed=42)
for p in [50,200]:
    s=sub_s.repartition(p); ts=time.time()
    KMeans(featuresCol='features',predictionCol='prediction',k=best_k,maxIter=10,seed=42).fit(s)
    strong_res.append({'partitions':p,'time':time.time()-ts})
    print(f"  {p} parts: {strong_res[-1]['time']:.2f}s")
strong_df=pd.DataFrame(strong_res)

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.plot(weak_df['records'],weak_df['time'],'bo-'); a1.set_xlabel('Records'); a1.set_ylabel('Time(s)'); a1.set_title('Weak Scaling')
a2.plot(strong_df['partitions'],strong_df['time'],'ro-'); a2.set_xlabel('Partitions'); a2.set_ylabel('Time(s)'); a2.set_title('Strong Scaling')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig5_scalability.png',dpi=150); plt.close()

fig,ax=plt.subplots(figsize=(8,5))
ax.boxplot([bsils,aris],labels=['Silhouette','ARI'])
ax.set_title('Stability Analysis — Bootstrap Results'); ax.set_ylabel('Score')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig5c_stability_boxplot.png',dpi=150); plt.close()
log_time("Scalability", t0)

print("\n" + "="*60)
print("STEP 12: PROFILES & VISUALS")
print("="*60)
t0 = time.time()

full_pred=km_best.transform(df_scaled)

profile=full_pred.groupBy('prediction').agg(*[F.mean(c).alias(f'mean_{c}') for c in cluster_features],F.count('*').alias('count')).orderBy('prediction')
profile_pd=profile.toPandas()
print(profile_pd.to_string())
profile_pd.to_csv('/content/crime_outputs/cluster_profiles.csv',index=False)

sizes=full_pred.groupBy('prediction').count().orderBy('prediction').toPandas()
fig,ax=plt.subplots(figsize=(8,8))
ax.pie(sizes['count'],labels=[f'Cluster {i}' for i in sizes['prediction']],autopct='%1.1f%%',colors=plt.cm.Set3.colors)
ax.set_title('Cluster Size Distribution')
plt.savefig('/content/crime_outputs/fig4a_cluster_sizes.png',dpi=150); plt.close()

sp=full_pred.select('prediction','Latitude','Longitude').sample(0.002,seed=42).toPandas()
fig,ax=plt.subplots(figsize=(10,12))
ax.scatter(sp['Longitude'],sp['Latitude'],c=sp['prediction'],cmap='Set2',alpha=0.3,s=3)
ax.set_title('Crime Clusters - Spatial'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig4b_spatial_clusters.png',dpi=150); plt.close()

from pyspark.ml.feature import PCA
ps=full_pred.sample(0.002,seed=42)
pca_pd=PCA(k=2,inputCol='features',outputCol='pca').fit(ps).transform(ps).select('prediction','pca').toPandas()
pca_pd['pc1']=pca_pd['pca'].apply(lambda x:float(x[0]))
pca_pd['pc2']=pca_pd['pca'].apply(lambda x:float(x[1]))
fig,ax=plt.subplots(figsize=(10,8))
ax.scatter(pca_pd['pc1'],pca_pd['pc2'],c=pca_pd['prediction'],cmap='Set2',alpha=0.3,s=5)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('PCA 2D Projection')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig4c_pca.png',dpi=150); plt.close()

centres=np.array(km_best.clusterCenters())
feat_imp_df=pd.DataFrame({'feature':cluster_features,'importance':np.std(centres,axis=0)}).sort_values('importance',ascending=True)
fig,ax=plt.subplots(figsize=(10,8))
ax.barh(feat_imp_df['feature'],feat_imp_df['importance'],color='steelblue')
ax.set_title('Feature Importance')
plt.tight_layout(); plt.savefig('/content/crime_outputs/fig4d_feature_importance.png',dpi=150); plt.close()
print("Saved: fig4a,4b,4c,4d")
log_time("Profiles", t0)

print("\n" + "="*60)
print("STEP 13: EXPORT")
print("="*60)
t0 = time.time()

missing_df.to_csv('/content/crime_outputs/tableau_data_quality.csv',index=False)
eval_table.to_csv('/content/crime_outputs/tableau_model_performance.csv',index=False)
elbow_df.to_csv('/content/crime_outputs/tableau_elbow.csv',index=False)
feat_imp_df.to_csv('/content/crime_outputs/tableau_feature_importance.csv',index=False)
year_counts.to_csv('/content/crime_outputs/tableau_year_trend.csv',index=False)
full_pred.select('prediction','Primary Type','Latitude','Longitude','HOUR','DAY_OF_WEEK',
    'MONTH_NUM','IS_WEEKEND','IS_NIGHT','IS_DOWNTOWN','SEVERITY','IS_VIOLENT',
    'ARREST_INT','DOMESTIC_INT','Year','District','Community Area'
).sample(0.01,seed=42).toPandas().to_csv('/content/crime_outputs/tableau_cluster_data.csv',index=False)
weak_df.to_csv('/content/crime_outputs/tableau_weak_scaling.csv',index=False)
strong_df.to_csv('/content/crime_outputs/tableau_strong_scaling.csv',index=False)
pd.DataFrame({'iteration':range(1,6),'silhouette':bsils,'ari':aris}).to_csv('/content/crime_outputs/tableau_stability.csv',index=False)

for f in sorted(os.listdir('/content/crime_outputs/')):
    print(f"  {f} ({os.path.getsize(f'/content/crime_outputs/{f}')/1024:.1f} KB)")
log_time("Export", t0)

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Dataset: {raw_count:,} records, {raw_cols} cols, {total_size/1024/1024/1024:.2f} GB")
print(f"Cleaned: {clean_count:,} | Features: {len(cluster_features)} | k={best_k}")
print(f"KMeans Sil={km_sil:.4f} | BKM Sil={bkm_sil:.4f} | GMM Sil={gmm_sil:.4f}")
print(f"sklearn Sil={sk_sil:.4f} | DBSCAN Sil={db_sil:.4f} | GPU Sil={gpu_sil:.4f}")
print(f"CI:[{ci_lo:.4f},{ci_hi:.4f}] ARI={m_ari:.4f}+/-{s_ari:.4f} | Speedup={speedup:.1f}x")
for k,v in results_log.items(): print(f"  {k}: {v}")
print("\nAll images:")
for img in ['fig1_eda','fig2_correlation','fig3_elbow_silhouette','fig4a_cluster_sizes',
            'fig4b_spatial_clusters','fig4c_pca','fig4d_feature_importance','fig5_scalability','fig5c_stability_boxplot']:
    print(f"  {img}.png")
print("\nDOWNLOAD: /content/crime_outputs/")

df_clean.unpersist(); df_scaled.unpersist()
train_df.unpersist(); val_df.unpersist(); test_df.unpersist()
spark.stop()
print("DONE")

STEP 1: GENERATE DATASET
Dataset: 0.77 GB, 7,900,000 records
[DONE] Data Generation: 97.50s

STEP 2: SPARK INGESTION
Raw: 7,900,000 records, 14 columns
Parquet: 260 MB
[DONE] Spark Ingestion: 108.75s

STEP 3: CLEANING & EDA
Cleaned: 7,900,000 -> 7,900,000 (dropped 0.0%)
Saved: fig1, fig2
[DONE] Cleaning & EDA: 72.35s

STEP 4: FEATURE ENGINEERING
Features (22)
[DONE] Feature Engineering: 1.60s

STEP 5: VECTORIZE & SCALE
Scaled: 3,118,379
[DONE] Vectorize & Scale: 304.62s

STEP 6: SPLIT
Train:2,169,100 Val:406,887 Test:542,392
Train sample:108,659 Val sample:40,613
[DONE] Split: 70.53s

STEP 7: CLUSTERING
--- Elbow ---
  k=4: Sil=0.1895
  k=5: Sil=0.1705
  k=6: Sil=0.1586
  k=7: Sil=0.1605
  k=8: Sil=0.1683
  k=9: Sil=0.1700
  k=10: Sil=0.1665
Best k=4
[DONE] Elbow: 78.06s
--- KMeans CV ---
KMeans: Sil=0.1776 k=3
[DONE] KMeans CV: 52.96s
--- BKM CV ---
BKM: Sil=0.1755
[DONE] BKM CV: 95.28s
--- GMM CV ---
GMM: Sil=0.1198
[DONE] GMM CV: 94.31s

STEP 8: SKLEARN
Sample: 47,005
sklearn: Sil=0